# DiffusionNet — Fracture Segmentation cho cổ vật 3D

Bản **sạch**, chạy tuần tự từ trên xuống. Mỗi vertex được phân loại:
- `0` = **original** (bề mặt gốc), `1` = **fracture** (bề mặt vỡ).

**Cải tiến so với baseline (IoU ~0.25):** input **HKS** (bất biến pose), loss **CE+Dice**, **k_eig=128**, **cache mesh+nhãn** (chạy lại nhanh), grad clipping.

**Trước khi chạy:** Settings → Accelerator = **GPU T4 x2 / P100**, Internet **ON**, Add Input dataset `breaking-bad-artifact-decompressed`.


## 1. Setup môi trường

In [ ]:
import os, sys, shutil, random, json, time
from pathlib import Path
from collections import deque, defaultdict

!pip install -q trimesh potpourri3d robust_laplacian plyfile plotly scikit-learn

DIFFNET_DIR = '/kaggle/working/diffusion-net'
if not os.path.exists(DIFFNET_DIR):
    !git clone https://github.com/nmwsharp/diffusion-net.git {DIFFNET_DIR}
sys.path.append(f'{DIFFNET_DIR}/src')

import numpy as np
import torch
import torch.nn.functional as F
import trimesh
from scipy.spatial import cKDTree
import diffusion_net

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ KHÔNG thấy GPU! Vào Settings → Accelerator → GPU T4 x2 / P100 rồi chạy lại.')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Cấu hình — chỉnh hết ở đây (để ablation)

Baseline cũ: `INPUT_FEATURES='xyz'`, `K_EIG=64`, `LOSS_MODE='ce'`.

In [ ]:
# Findings:
#  - HKS chuẩn hóa fix loss bùng nổ; weight [1,3] cân P/R.
#  - HKS đơn thuần quá trơn (IoU ~0.30); thêm CURVATURE -> IoU 0.42.
#  - Overfit test: hks 0.45 -> xyz_hks_curv 0.63 (còn lên) => feature ĐỦ, còn dư địa capacity.
#  - => RUN NÂNG CAO: C_WIDTH=256 + full-set + 25 epoch. Kỳ vọng val ~0.45-0.50.

INPUT_FEATURES   = 'xyz_hks_curv'  # ghép tự do: 'xyz','hks','curv' (vd 'hks' | 'xyz_hks' | 'xyz_hks_curv')
N_HKS            = 16
K_EIG            = 128
C_WIDTH          = 256         # 128->256: overfit còn lên -> tăng capacity (params ~1.8M, vẫn nhẹ)
N_BLOCK          = 4

LOSS_MODE        = 'ce_dice'   # 'ce' | 'ce_dice' | 'focal'
CE_CLASS_WEIGHTS = [1.0, 3.0]  # [original, fracture]
DICE_WEIGHT      = 1.0

N_EPOCHS          = 25
SAMPLES_PER_EPOCH = None        # None = FULL train set/epoch (epoch 1 chậm vì dựng cache, sau ~5'/epoch)
VAL_SAMPLES       = None         # None = full val mỗi epoch (chọn best chính xác; val chỉ ~400 mesh)
LR                = 1e-3
GRAD_CLIP         = 1.0
LABEL_THRESHOLD_RATIO = 0.005

# C_in suy từ feature flags
C_IN = (3 if 'xyz' in INPUT_FEATURES else 0) + (N_HKS if 'hks' in INPUT_FEATURES else 0) + (1 if 'curv' in INPUT_FEATURES else 0)
assert C_IN > 0, "INPUT_FEATURES phải chứa ít nhất 1 trong: 'xyz','hks','curv'"
print(f'features={INPUT_FEATURES}  C_in={C_IN}  k_eig={K_EIG}  loss={LOSS_MODE}  '
      f'weights={CE_CLASS_WEIGHTS}  width={C_WIDTH}  epochs={N_EPOCHS}  samples/epoch={SAMPLES_PER_EPOCH}')

## 3. Tự động dò đường dẫn dataset

In [ ]:
INPUT_BASE = Path('/kaggle/input')
print('Datasets đã attach:')
for d in INPUT_BASE.iterdir():
    print(f'  - {d.name}')
print()

def find_artifact_root(base, max_depth=5):
    """Tìm folder chứa nhiều thư mục con *_sf/."""
    best, best_count = None, 0
    queue = deque([(base, 0)])
    while queue:
        path, depth = queue.popleft()
        if depth > max_depth:
            continue
        try:
            children = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        sf_count = sum(1 for c in children if c.is_dir() and c.name.endswith('_sf'))
        if sf_count > best_count:
            best_count = sf_count; best = path
        for c in children:
            if c.is_dir() and not c.name.endswith('_sf'):
                queue.append((c, depth + 1))
    return best

def find_file_anywhere(base, name, max_depth=6):
    queue = deque([(base, 0)])
    while queue:
        path, depth = queue.popleft()
        if depth > max_depth:
            continue
        try:
            children = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        for c in children:
            if c.is_file() and c.name == name:
                return c
            if c.is_dir():
                queue.append((c, depth + 1))
    return None

ARTIFACT_ROOT = find_artifact_root(INPUT_BASE)
train_txt = find_file_anywhere(INPUT_BASE, 'artifact.train.txt')
val_txt   = find_file_anywhere(INPUT_BASE, 'artifact.val.txt')
assert ARTIFACT_ROOT is not None, '❌ Không tìm thấy artifact data (*_sf folders)'
assert train_txt is not None and val_txt is not None, '❌ Không tìm thấy split files'

with open(train_txt) as fp:
    train_ids = [l.strip().split('/')[-1] for l in fp if l.strip()]
with open(val_txt) as fp:
    val_ids = [l.strip().split('/')[-1] for l in fp if l.strip()]
print(f'✅ ARTIFACT_ROOT = {ARTIFACT_ROOT}')
print(f'Train objects: {len(train_ids)} | Val objects: {len(val_ids)}')

## 4. Dataset (có cache mesh + nhãn)

Lần đầu mỗi sample: đọc các `piece_*.obj`, sinh nhãn (vertex gần piece khác = fracture), normalize, rồi **lưu cache `.npz`**. Lần sau (epoch 2+ hoặc chạy lại notebook): **load thẳng cache** → bỏ qua đọc obj + KDTree, giảm mạnh tải CPU.

Operator (Laplacian/eigen) được cache riêng bởi `op_cache`. Cache nhãn có gắn `threshold` trong tên file → đổi `LABEL_THRESHOLD_RATIO` sẽ tự sinh cache mới (không lấy nhãn cũ).

In [ ]:
from torch.utils.data import Dataset

class BBFractureDataset(Dataset):
    """1 sample = 1 fracture (1 cổ vật vỡ nhiều mảnh)."""
    def __init__(self, data_root, object_ids, op_cache_dir, mesh_cache_dir,
                 label_threshold_ratio=0.005, max_vertices=30000, k_eig=128):
        self.data_root = Path(data_root)
        self.op_cache_dir = Path(op_cache_dir); self.op_cache_dir.mkdir(parents=True, exist_ok=True)
        self.mesh_cache_dir = Path(mesh_cache_dir); self.mesh_cache_dir.mkdir(parents=True, exist_ok=True)
        self.label_threshold_ratio = label_threshold_ratio
        self.max_vertices = max_vertices
        self.k_eig = k_eig
        self.samples = []
        for obj_id in object_ids:
            obj_dir = self.data_root / obj_id
            if not obj_dir.exists():
                continue
            for frac_dir in sorted(obj_dir.iterdir()):
                if frac_dir.is_dir() and any(frac_dir.glob('piece_*.obj')):
                    self.samples.append((obj_id, frac_dir.name))
        print(f'  Dataset: {len(self.samples)} fractures từ {len(object_ids)} objects')

    def __len__(self):
        return len(self.samples)

    def _process_raw(self, frac_dir):
        """Đọc obj + sinh nhãn + normalize. Trả (verts, faces, labels) hoặc None."""
        piece_files = sorted(frac_dir.glob('piece_*.obj'))
        pieces = []
        for pf in piece_files:
            m = trimesh.load(str(pf), process=False)
            pieces.append((np.asarray(m.vertices, dtype=np.float32),
                           np.asarray(m.faces, dtype=np.int64)))
        if len(pieces) == 0:
            return None
        all_verts = np.concatenate([v for v, _ in pieces])
        bbox_diag = np.linalg.norm(all_verts.max(0) - all_verts.min(0))
        threshold = bbox_diag * self.label_threshold_ratio
        merged_v, merged_f, merged_labels = [], [], []
        offset = 0
        for i, (v, f) in enumerate(pieces):
            other = [pv for j, (pv, _) in enumerate(pieces) if j != i]
            if other:
                dists, _ = cKDTree(np.concatenate(other)).query(v, k=1)
                labels_i = (dists < threshold).astype(np.int64)
            else:
                labels_i = np.zeros(len(v), dtype=np.int64)
            merged_v.append(v); merged_f.append(f + offset); merged_labels.append(labels_i)
            offset += len(v)
        verts = np.concatenate(merged_v)
        faces = np.concatenate(merged_f).astype(np.int64)
        labels = np.concatenate(merged_labels).astype(np.int64)
        verts = verts - verts.mean(0)
        scale = np.linalg.norm(verts.max(0) - verts.min(0))
        if scale > 0:
            verts = verts / scale
        return verts.astype(np.float32), faces, labels

    def _get_mesh(self, obj_id, frac_name):
        key = f'{obj_id}__{frac_name}__t{self.label_threshold_ratio}'
        cache_f = self.mesh_cache_dir / f'{key}.npz'
        if cache_f.exists():
            d = np.load(cache_f)
            return d['verts'], d['faces'], d['labels']
        res = self._process_raw(self.data_root / obj_id / frac_name)
        if res is None:
            return None
        verts, faces, labels = res
        np.savez(cache_f, verts=verts, faces=faces, labels=labels)
        return verts, faces, labels

    def __getitem__(self, idx):
        obj_id, frac_name = self.samples[idx]
        res = self._get_mesh(obj_id, frac_name)
        if res is None:
            return None
        verts_np, faces_np, labels_np = res
        if len(verts_np) > self.max_vertices:
            return None
        verts = torch.tensor(verts_np, dtype=torch.float32)
        faces = torch.tensor(faces_np, dtype=torch.long)
        labels = torch.tensor(labels_np, dtype=torch.long)
        try:
            frames, mass, L, evals, evecs, gradX, gradY = diffusion_net.geometry.get_operators(
                verts, faces, k_eig=self.k_eig, op_cache_dir=str(self.op_cache_dir))
        except Exception as e:
            print(f'⚠️ Operator fail {obj_id}/{frac_name}: {e}')
            return None
        return {'verts': verts, 'faces': faces, 'labels': labels,
                'mass': mass, 'L': L, 'evals': evals, 'evecs': evecs,
                'gradX': gradX, 'gradY': gradY, 'obj_id': obj_id, 'frac_name': frac_name}

# QUAN TRỌNG: cache để ở /kaggle/temp (scratch ephemeral) — KHÔNG tính vào 20GB output,
# KHÔNG bị snapshot khi Save Version. op_cache có thể 20-40GB nên bắt buộc để ở đây.
# Chỉ /kaggle/working mới bị giới hạn 20GB (ta chỉ lưu checkpoint vài MB ở đó).
OP_CACHE   = '/kaggle/temp/op_cache'
MESH_CACHE = '/kaggle/temp/mesh_cache'
print('Init train dataset...')
train_ds = BBFractureDataset(ARTIFACT_ROOT, train_ids, OP_CACHE, MESH_CACHE,
                             label_threshold_ratio=LABEL_THRESHOLD_RATIO, k_eig=K_EIG)
print('Init val dataset...')
val_ds = BBFractureDataset(ARTIFACT_ROOT, val_ids, OP_CACHE, MESH_CACHE,
                           label_threshold_ratio=LABEL_THRESHOLD_RATIO, k_eig=K_EIG)

## 5. Test load 1 sample + visualize nhãn

Phải nhìn thấy vùng **đỏ** (fracture) đúng vị trí đường vỡ.

In [ ]:
print('Loading sample 0 (lần đầu tính operator nên chậm)...')
t0 = time.time()
sample = train_ds[0]
print(f'Loaded trong {time.time()-t0:.1f}s')
if sample is None:
    print('❌ Sample = None — kiểm tra data.')
else:
    n = len(sample['labels']); n_frac = int((sample['labels'] == 1).sum())
    print(f"obj={sample['obj_id']}  frac={sample['frac_name']}")
    print(f"Vertices={tuple(sample['verts'].shape)}  Faces={tuple(sample['faces'].shape)}")
    print(f"Fracture: {n_frac}/{n} ({100*n_frac/n:.1f}%)  |  Original: {n-n_frac} ({100*(n-n_frac)/n:.1f}%)")

In [ ]:
import plotly.graph_objects as go

v = sample['verts'].numpy(); f = sample['faces'].numpy(); labels = sample['labels'].numpy()
fig = go.Figure(data=[go.Mesh3d(
    x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
    intensity=labels.astype(np.float32),
    colorscale=[[0, 'lightblue'], [1, 'red']], showscale=True, cmin=0, cmax=1, flatshading=True)])
fig.update_layout(title=f"{sample['obj_id']} / {sample['frac_name']} — GT (xanh=original, đỏ=fracture)",
                  scene=dict(aspectmode='data'), margin=dict(l=0, r=0, t=60, b=0), width=800, height=600)
fig.show()

## 6. Model + Loss

In [ ]:
model = diffusion_net.layers.DiffusionNet(
    C_in=C_IN, C_out=2, C_width=C_WIDTH, N_block=N_BLOCK,
    outputs_at='vertices', dropout=True).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}  '
      f'(C_in={C_IN}, width={C_WIDTH}, blocks={N_BLOCK})')

ce_weights = torch.tensor(CE_CLASS_WEIGHTS, device=device)
ce_fn = torch.nn.CrossEntropyLoss(weight=ce_weights)

def dice_loss(logits, labels, eps=1e-6):
    probs = F.softmax(logits, dim=-1)[:, 1]
    target = (labels == 1).float()
    inter = (probs * target).sum()
    return 1.0 - (2.0 * inter + eps) / (probs.sum() + target.sum() + eps)

def focal_loss(logits, labels, gamma=2.0):
    logpt = F.log_softmax(logits, dim=-1)
    pt = logpt.exp()
    logpt = logpt.gather(1, labels.unsqueeze(1)).squeeze(1)
    pt = pt.gather(1, labels.unsqueeze(1)).squeeze(1)
    at = ce_weights.gather(0, labels)
    return (at * (1 - pt) ** gamma * (-logpt)).mean()

def compute_loss(logits, labels):
    if LOSS_MODE == 'ce':
        return ce_fn(logits, labels)
    if LOSS_MODE == 'ce_dice':
        return ce_fn(logits, labels) + DICE_WEIGHT * dice_loss(logits, labels)
    if LOSS_MODE == 'focal':
        return focal_loss(logits, labels)
    raise ValueError(f'LOSS_MODE không hợp lệ: {LOSS_MODE}')

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
print(f'Loss={LOSS_MODE} | class weights={CE_CLASS_WEIGHTS} | lr={LR} | grad_clip={GRAD_CLIP}')

## 7. Hàm train / eval

In [ ]:
def compute_metrics(preds, labels):
    """Acc, Precision, Recall, F1, IoU cho lớp fracture (1)."""
    preds = preds.flatten(); labels = labels.flatten()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    total = tp + fp + fn + tn
    acc = (tp + tn) / total if total else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0
    return {'acc': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'iou': iou}

def _std_log(x):
    """log-compress + standardize từng kênh (theo vertex của 1 mesh)."""
    x = torch.log(x.clamp_min(1e-8))
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + 1e-6)

def get_features(sample):
    """Ghép feature theo INPUT_FEATURES ('xyz','hks','curv'):
       - xyz : tọa độ đã normalize (local, biến thiên sắc).
       - hks : Heat Kernel Signature (nội tại, bất biến pose) — nhưng TRƠN.
       - curv: |mean curvature| = |M^-1 L x| — cao ở mặt vỡ gồ ghề, SẮC ở biên."""
    verts = sample['verts'].to(device)
    feats = []
    if 'xyz' in INPUT_FEATURES:
        feats.append(verts)
    if 'hks' in INPUT_FEATURES:
        hks = diffusion_net.geometry.compute_hks_autoscale(
            sample['evals'].to(device), sample['evecs'].to(device), N_HKS)
        feats.append(_std_log(hks))   # HKS thô ~10^5-10^6 -> bắt buộc chuẩn hóa
    if 'curv' in INPUT_FEATURES:
        L = sample['L'].to(device); mass = sample['mass'].to(device)
        Hvec = torch.sparse.mm(L, verts) / mass.clamp_min(1e-8).unsqueeze(-1)
        curv = Hvec.norm(dim=-1, keepdim=True)
        feats.append(_std_log(curv))
    return feats[0] if len(feats) == 1 else torch.cat(feats, dim=-1)

def forward_sample(model, sample):
    return model(
        x_in=get_features(sample),
        mass=sample['mass'].to(device), L=sample['L'].to(device),
        evals=sample['evals'].to(device), evecs=sample['evecs'].to(device),
        gradX=sample['gradX'].to(device), gradY=sample['gradY'].to(device),
        faces=sample['faces'].to(device))

def train_epoch(model, dataset, n_samples, optimizer):
    model.train()
    if n_samples is None:
        indices = list(range(len(dataset))); random.shuffle(indices)
    else:
        indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    losses, all_preds, all_labels = [], [], []
    for i, idx in enumerate(indices):
        sample = dataset[idx]
        if sample is None:
            continue
        optimizer.zero_grad()
        logits = forward_sample(model, sample)
        labels = sample['labels'].to(device)
        loss = compute_loss(logits, labels)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
        all_preds.append(logits.argmax(dim=-1).cpu()); all_labels.append(sample['labels'])
        if (i + 1) % 50 == 0:
            print(f'    train {i+1}/{len(indices)} | loss={np.mean(losses[-50:]):.4f}')
    m = compute_metrics(torch.cat(all_preds), torch.cat(all_labels)); m['loss'] = float(np.mean(losses))
    return m

@torch.no_grad()
def eval_epoch(model, dataset, n_samples=None):
    model.eval()
    if n_samples is None:
        indices = list(range(len(dataset)))
    else:
        indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    losses, all_preds, all_labels = [], [], []
    for idx in indices:
        sample = dataset[idx]
        if sample is None:
            continue
        logits = forward_sample(model, sample)
        labels = sample['labels'].to(device)
        losses.append(compute_loss(logits, labels).item())
        all_preds.append(logits.argmax(dim=-1).cpu()); all_labels.append(sample['labels'])
    m = compute_metrics(torch.cat(all_preds), torch.cat(all_labels)); m['loss'] = float(np.mean(losses))
    return m

## 8. Train

Epoch 1 chậm (đang dựng op_cache + mesh_cache). Từ epoch 2 sẽ nhanh hơn hẳn và GPU được dùng nhiều hơn.

In [ ]:
history = {'train': [], 'val': []}
best_val_iou = 0
best_state = None

for epoch in range(N_EPOCHS):
    print(f'\n=== Epoch {epoch+1}/{N_EPOCHS} ===')
    t0 = time.time()
    train_m = train_epoch(model, train_ds, SAMPLES_PER_EPOCH, optimizer)
    val_m = eval_epoch(model, val_ds, VAL_SAMPLES)
    scheduler.step()
    history['train'].append(train_m); history['val'].append(val_m)
    print(f'  TRAIN | loss={train_m["loss"]:.4f} acc={train_m["acc"]:.3f} '
          f'P={train_m["precision"]:.3f} R={train_m["recall"]:.3f} F1={train_m["f1"]:.3f} IoU={train_m["iou"]:.3f}')
    print(f'  VAL   | loss={val_m["loss"]:.4f} acc={val_m["acc"]:.3f} '
          f'P={val_m["precision"]:.3f} R={val_m["recall"]:.3f} F1={val_m["f1"]:.3f} IoU={val_m["iou"]:.3f}')
    print(f'  Time={time.time()-t0:.1f}s  lr={optimizer.param_groups[0]["lr"]:.1e}')
    if val_m['iou'] > best_val_iou:
        best_val_iou = val_m['iou']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  ⭐ Best val IoU = {best_val_iou:.3f}')

print(f'\n🎉 Done. Best val IoU: {best_val_iou:.3f}')

## 9. Đường cong loss / metrics

In [ ]:
import matplotlib.pyplot as plt

ep = list(range(1, len(history['train']) + 1))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, key, title in zip(ax, ['loss', 'f1', 'iou'], ['Loss', 'F1 (fracture)', 'IoU (fracture)']):
    a.plot(ep, [m[key] for m in history['train']], 'o-', label='train')
    a.plot(ep, [m[key] for m in history['val']], 's-', label='val')
    a.set_title(title); a.set_xlabel('Epoch'); a.legend(); a.grid(True)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Đánh giá đầy đủ trên toàn bộ val (best checkpoint)

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)
print('Evaluating trên TOÀN BỘ val set...')
final_metrics = eval_epoch(model, val_ds, n_samples=None)
print('\n=== FINAL METRICS (Val) ===')
for k, val in final_metrics.items():
    print(f'  {k:12s}: {val:.4f}')

## 11. Visualize prediction (vài sample val)

In [ ]:
import plotly.graph_objects as go   # tự import để cell chạy độc lập (không phụ thuộc mục 5)

@torch.no_grad()
def visualize_prediction(sample, model):
    model.eval()
    pred = forward_sample(model, sample).argmax(dim=-1).cpu().numpy()
    v = sample['verts'].numpy(); f = sample['faces'].numpy(); gt = sample['labels'].numpy()
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
        intensity=gt.astype(np.float32), colorscale=[[0,'lightblue'],[1,'red']], cmin=0, cmax=1, visible=True))
    fig.add_trace(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
        intensity=pred.astype(np.float32), colorscale=[[0,'lightblue'],[1,'red']], cmin=0, cmax=1, visible=False))
    fig.update_layout(title=f"{sample['obj_id']} / {sample['frac_name']}",
        updatemenus=[dict(buttons=[
            dict(label='Ground Truth', method='update', args=[{'visible':[True, False]}]),
            dict(label='Prediction', method='update', args=[{'visible':[False, True]}])],
            direction='down', showactive=True, x=0.1, y=1.15)],
        scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=80,b=0), width=800, height=600)
    fig.show()
    m = compute_metrics(torch.tensor(pred), torch.tensor(gt))
    print(f"  F1={m['f1']:.3f}  IoU={m['iou']:.3f}  Acc={m['acc']:.3f}")

for idx in [0, 5, 10]:
    s = val_ds[idx]
    if s is not None:
        print(f'--- val sample {idx} ---')
        visualize_prediction(s, model)

## 12. Lưu checkpoint + metrics

In [ ]:
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
CONFIG = {'input_features': INPUT_FEATURES, 'n_hks': N_HKS, 'C_in': C_IN, 'k_eig': K_EIG,
          'C_width': C_WIDTH, 'N_block': N_BLOCK, 'loss_mode': LOSS_MODE,
          'ce_class_weights': CE_CLASS_WEIGHTS, 'dice_weight': DICE_WEIGHT, 'lr': LR,
          'n_epochs': N_EPOCHS, 'samples_per_epoch': SAMPLES_PER_EPOCH,
          'label_threshold_ratio': LABEL_THRESHOLD_RATIO}
torch.save({'model_state_dict': best_state if best_state else model.state_dict(),
            'best_val_iou': best_val_iou, 'config': CONFIG}, f'{CKPT_DIR}/diffnet_best.pt')
with open(f'{CKPT_DIR}/history.json', 'w') as fp: json.dump(history, fp, indent=2)
with open(f'{CKPT_DIR}/final_metrics.json', 'w') as fp: json.dump(final_metrics, fp, indent=2)
with open(f'{CKPT_DIR}/config.json', 'w') as fp: json.dump(CONFIG, fp, indent=2)
print('✅ Saved checkpoints:')
!ls -lah {CKPT_DIR}/

# Kiểm tra dung lượng: /kaggle/working PHẢI nhỏ (< 20GB), cache nằm ở /kaggle/temp (không tính)
print('\n📦 /kaggle/working (OUTPUT — giới hạn 20GB, đây mới là cái được Save):')
!du -sh /kaggle/working 2>/dev/null; du -sh /kaggle/working/* 2>/dev/null
print('\n🗑️ /kaggle/temp (scratch — KHÔNG tính vào output, sẽ bị xóa khi hết session):')
!du -sh /kaggle/temp/* 2>/dev/null

## 13. (Chẩn đoán) Overfit test — feature có đủ mạnh không?

Chạy cell dưới để kiểm tra model có học thuộc nổi 30 mesh hay không. Dùng để quyết định trần IoU đến từ **feature/nhãn** hay từ **dữ liệu/generalization**. Đổi `INPUT_FEATURES` ở cell 2 (vd `'hks'` vs `'xyz_hks_curv'`) rồi chạy lại cell 2 + cell này để so sánh.

In [ ]:
# Chẩn đoán: model có HỌC THUỘC nổi 30 mesh cố định không? (test feature đủ mạnh chưa)
# Chạy ĐỘC LẬP sau khi đã chạy cell 2,4,6,7 (cần C_IN, train_ds, forward_sample, compute_loss).
small_idx = list(range(30))
ofit = diffusion_net.layers.DiffusionNet(C_in=C_IN, C_out=2, C_width=C_WIDTH,
        N_block=N_BLOCK, outputs_at='vertices', dropout=False).to(device)
opt = torch.optim.Adam(ofit.parameters(), lr=1e-3)
for ep in range(40):
    ofit.train(); preds = []; labs = []
    for idx in small_idx:
        s = train_ds[idx]
        if s is None:
            continue
        opt.zero_grad()
        logits = forward_sample(ofit, s)
        compute_loss(logits, s['labels'].to(device)).backward(); opt.step()
        preds.append(logits.argmax(-1).cpu()); labs.append(s['labels'])
    if (ep + 1) % 5 == 0:
        m = compute_metrics(torch.cat(preds), torch.cat(labs))
        print(f'overfit ep{ep+1}: IoU={m["iou"]:.3f} P={m["precision"]:.3f} R={m["recall"]:.3f}')
# Đọc: IoU -> 0.7+  = feature ĐỦ MẠNH (trần do data/generalization).
#      IoU kẹt ~0.45 = feature/nhãn yếu (đổi feature hoặc cách sinh nhãn).

In [ ]:
import plotly.graph_objects as go

def _components(faces, n):
    """Union-find: id thành phần liên thông của mỗi vertex (= từng mảnh vỡ)."""
    parent = np.arange(n)
    def find(x):
        root = x
        while parent[root] != root: root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root
    for a, b, c in faces:
        for u, w in ((int(a), int(b)), (int(b), int(c))):
            ru, rw = find(u), find(w)
            if ru != rw: parent[ru] = rw
    return np.array([find(i) for i in range(n)])

@torch.no_grad()
def simulate(sample, model, explode=0.35, show='pred'):
    """Dự đoán fracture + TÁCH các mảnh ra để lộ mặt cắt. show='pred'|'gt'."""
    model.eval()
    pred = forward_sample(model, sample).argmax(-1).cpu().numpy()
    gt   = sample['labels'].numpy()
    v = sample['verts'].numpy().copy(); f = sample['faces'].numpy()
    comp = _components(f, len(v)); ids = np.unique(comp); gc = v.mean(0)
    if len(ids) > 1:                          # đẩy từng mảnh ra xa tâm
        for cid in ids:
            mk = comp == cid
            d = v[mk].mean(0) - gc; nrm = np.linalg.norm(d)
            if nrm > 1e-6:
                v[mk] += (d / nrm) * explode
    color = (pred if show == 'pred' else gt).astype(np.float32)
    m = compute_metrics(torch.tensor(pred), torch.tensor(gt))
    fig = go.Figure(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
        intensity=color, colorscale=[[0,'lightgray'],[1,'crimson']],
        cmin=0, cmax=1, flatshading=True, showscale=False))
    fig.update_layout(width=820, height=620, scene=dict(aspectmode='data'),
        title=f"{sample.get('obj_id','')} — {'Prediction' if show=='pred' else 'Ground Truth'} "
              f"(đỏ=fracture), {len(ids)} mảnh, IoU={m['iou']:.3f} F1={m['f1']:.3f}")
    fig.show()

# Mô phỏng: tách mảnh + tô đỏ vùng model dự đoán là fracture. show='gt' để xem nhãn thật.
for idx in [0, 5, 10]:
    s = val_ds[idx]
    if s is None: continue
    print(f"--- val {idx}: {s['obj_id']} / {s['frac_name']} ---")
    simulate(s, model, explode=0.35, show='pred')